<a href="https://colab.research.google.com/github/demichie/Principles-of-Numerical-Modelling-in-Geosciences/blob/main/Chapter7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 7: Numerical Solution of the Linear Advection Equation

This notebook contains the Python code examples for Chapter 7 of "Principles of Numerical Modelling in Geosciences".

Having developed a solver for the parabolic Heat Equation, we now turn our attention to another cornerstone of transport phenomena: **advection**. This chapter is dedicated to understanding the simplest mathematical representation of this process—the 1D Linear Advection Equation—and to developing and analysing numerical schemes for its solution, focusing on concepts like stability (the CFL condition) and numerical artifacts (diffusion and dispersion).

## 7.1 The Linear Advection Equation: Physical and Mathematical Basis

### 7.1.3 Analytical Solution and Visualization with Python

The 1D linear advection equation, $\frac{\partial \phi}{\partial t} + u \frac{\partial \phi}{\partial x} = 0$, has a simple and elegant analytical solution: an initial profile $\phi_0(x)$ is simply translated by a distance $ut$ without changing its shape, such that $\phi(x,t) = \phi_0(x - ut)$.

The following script visualizes this exact, distortion-free translation, which will serve as a critical reference for our numerical methods.

In [ ]:
# --- Analytical Solution Visualization ---
import numpy as np
import matplotlib.pyplot as plt

def initialConditionGaussian(x, x0=0.5, sigma=0.1): # Note: camelCase for consistency
    """A Gaussian pulse as an initial condition."""
    return np.exp(-0.5 * ((x - x0) / sigma)**2)

def initialConditionSquareWave(x, xStart=0.25, xEnd=0.75, phiLow=0.0, phiHigh=1.0): # camelCase
    """A square wave (top-hat) as an initial condition."""
    phi0 = np.zeros_like(x) # Using np.zeros_like for efficiency
    phi0[(x >= xStart) & (x <= xEnd)] = phiHigh
    return phi0

# Domain and parameters for analytical solution
lengthDomain = 2.0
nxAnalytical = 201 # Using camelCase for variable name
xAnalytical = np.linspace(0, lengthDomain, nxAnalytical)
advectionVelocity = 1.0    # Advection velocity

# Choose initial condition type
# icFunction = initialConditionGaussian # Using camelCase
icFunction = initialConditionSquareWave # Using camelCase

# Initial profile
# For Square Wave:
phi0Analytical = icFunction(xAnalytical, xStart=0.25, xEnd=0.75) # camelCase
# For Gaussian specific parameters:
# phi0Analytical = icFunction(xAnalytical, x0=0.5, sigma=0.08)


# Time points for plotting
timeInitial = 0.0
timeFinalAnalytical = 0.5

# Analytical solution at tFinal
xShiftedAnalytical = xAnalytical - advectionVelocity * timeFinalAnalytical
# For Square Wave:
phiTFinalAnalytical = icFunction(xShiftedAnalytical, xStart=0.25, xEnd=0.75) # camelCase
# For Gaussian:
# phiTFinalAnalytical = icFunction(xShiftedAnalytical, x0=0.5, sigma=0.08)


# Plotting
plt.figure(figsize=(10, 6))
plt.plot(xAnalytical, phi0Analytical, 'b-',
         label=f'Initial Condition $\\phi_0(x)$ (t={timeInitial:.1f})')
plt.plot(xAnalytical, phiTFinalAnalytical, 'r--',
         label=f'Exact Solution $\\phi(x,t)$ (t={timeFinalAnalytical:.1f}, u={advectionVelocity})')
plt.xlabel('Position x')
plt.ylabel('$\\phi$')
plt.title('Exact Solution of 1D Linear Advection')
plt.legend()
plt.grid(True)
plt.ylim(-0.1, 1.1)
plt.show()

## 7.3 Finite Difference Schemes for Linear Advection

### 7.3.2 Scheme 1: Forward Time, Centered Space (FTCS)

The FTCS scheme uses a Forward Difference in Time and a Centered Difference in Space. Its update formula is:
$$ \phi_i^{n+1} = \phi_i^n - \frac{C}{2} (\phi_{i+1}^n - \phi_{i-1}^n) $$
where $C$ is the Courant number. The following script implements this scheme with periodic boundary conditions.

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt

def initialConditionSquareWave(x, xStart=0.25, xEnd=0.75, phiLow=0.0, phiHigh=1.0):
    phi0 = np.full_like(x, phiLow)
    phi0[(x >= xStart) & (x <= xEnd)] = phiHigh
    return phi0

def analyticalSolutionPeriodic(xGrid, t, u, L, phi0Func, xStartIC, xEndIC):
    xEffectiveSource = (xGrid - u * t) % L
    return phi0Func(xEffectiveSource, xStart=xStartIC, xEnd=xEndIC)

# --- 1. Simulation Parameters ---
lengthDomain = 1.0
numXPoints = 101
dxStep = lengthDomain / (numXPoints -1) if numXPoints > 1 else lengthDomain
advectionVelocity = 1.0

courantNumber = 0.40 # Courant number (FTCS is unstable, but we test a value)
deltaTime = courantNumber * dxStep / abs(advectionVelocity) if abs(advectionVelocity) > 1e-9 else 1e-3
timeFinal = 0.050  # Short time to see instability develop
numTimeSteps = int(timeFinal / deltaTime) if deltaTime > 0 else 0

print(f"FTCS - Domain: L={lengthDomain}, N_x={numXPoints}, dx={dxStep:.4f}")
print(f"FTCS - Velocity: u={advectionVelocity}, CFL={courantNumber:.2f}, dt={deltaTime:.4e}, Time={timeFinal:.3f}, Steps={numTimeSteps}")

# --- 2. Grid and Initial Condition ---
xGridFTCS = np.linspace(0, lengthDomain, numXPoints, endpoint=True)
xInitialStartFTCS = 0.15
xInitialEndFTCS = 0.35
phiInitialFTCS = initialConditionSquareWave(xGridFTCS, xStart=xInitialStartFTCS, xEnd=xInitialEndFTCS)

# Analytical solution at tFinal for comparison
phiAnalyticalFinalFTCS = analyticalSolutionPeriodic(xGridFTCS, timeFinal, advectionVelocity,
                                                 lengthDomain, initialConditionSquareWave,
                                                 xStartIC=xInitialStartFTCS, xEndIC=xInitialEndFTCS)

# --- 3. Initialization for FTCS ---
phiCurrent = phiInitialFTCS.copy()    # Current solution array
phiNew = np.zeros_like(phiCurrent)    # Array for the solution at the next time step

# --- 4. Time-stepping Loop (FTCS with custom Periodic BCs) ---
if numTimeSteps > 0:
    # Define the number of unique intervals for modulo arithmetic
    num_unique_intervals = numXPoints - 1
    if num_unique_intervals == 0: num_unique_intervals = 1 # Avoid error for single point grid

    for nLoopIndex in range(numTimeSteps):
        phiOldStep = phiCurrent.copy() # Values from time level 'n' for RHS

        # Loop over ALL computational nodes from 0 to Nx-2
        for iNode in range(numXPoints -1):
            # Determine periodic indices for neighbors based on the defined logic
            iMinus1 = (iNode - 1) % num_unique_intervals
            iPlus1 = (iNode + 1) % num_unique_intervals

            # FTCS formula using values from phiOldStep
            phiNew[iNode] = phiOldStep[iNode] - \
                            courantNumber / 2.0 * \
                            (phiOldStep[iPlus1] - phiOldStep[iMinus1])

        # Synchronization step
        phiNew[numXPoints-1] = phiNew[0]
        phiCurrent = phiNew.copy() # Update solution for the next time step
else:
    phiCurrent = phiInitialFTCS.copy()

# --- 5. Plotting Results ---
plt.figure(figsize=(10, 6))
plt.plot(xGridFTCS, phiInitialFTCS, label='Initial Condition (t=0)', linestyle=':', color='gray', lw=2)
plt.plot(xGridFTCS, phiCurrent, label=f'FTCS Numerical (t={timeFinal:.3f}, CFL={courantNumber:.2f})',
         marker='.', markersize=5, linestyle='-')
plt.plot(xGridFTCS, phiAnalyticalFinalFTCS, label=f'Analytical (t={timeFinal:.3f})',
         linestyle='--', color='red', lw=2)

plt.title(f'1D Linear Advection with FTCS Scheme (Periodic BCs)')
plt.xlabel('Spatial domain x')
plt.ylabel('$\\phi(x,t)$')
plt.legend()
plt.grid(True)
plt.ylim(-1.2, 2.2)
plt.show()

if numTimeSteps > 0 and (np.any(np.isnan(phiCurrent)) or np.any(np.abs(phiCurrent) > 2.5*np.max(np.abs(phiInitialFTCS)) if np.max(np.abs(phiInitialFTCS)) > 1e-9 else 1.0 )):
    print("Instability strongly suggested by FTCS solution (NaNs or large growth)!")
else:
    print("FTCS simulation completed. Visually inspect plot for stability.")

**Initial Observation:** As clearly visible in the plot, the FTCS scheme is **unconditionally unstable** for the pure advection problem, producing a solution riddled with rapidly growing, non-physical oscillations. This dramatic failure motivates the search for alternative, stable numerical schemes.

### 7.3.3 Scheme 2: Upwind Schemes - Aligning with the Flow Direction

The failure of FTCS highlights a critical point: for hyperbolic problems, schemes must respect the direction of information flow. This motivates **upwind schemes**, which use a one-sided difference taken from the "upwind" (upstream) direction.

#### Forward Time, Backward Space (FTBS) Scheme (for $u>0$)

For positive velocity ($u>0$), we use a Backward Difference in Space. The update formula is:
$$ \phi_i^{n+1} = (1-C)\phi_i^n + C \phi_{i-1}^n $$
This scheme is first-order accurate and, as we will see, conditionally stable.

In [ ]:
# --- Upwind (FTBS for u > 0) Scheme Implementation and Test ---
import numpy as np
import matplotlib.pyplot as plt

# Functions are redefined here to make the cell self-contained
def initialConditionSquareWave(x, xStart=0.25, xEnd=0.75, phiLow=0.0, phiHigh=1.0):
    phi0 = np.full_like(x, phiLow)
    phi0[(x >= xStart) & (x <= xEnd)] = phiHigh
    return phi0

def analyticalSolutionPeriodic(xGrid, t, u, L, phi0Func, xStartIC, xEndIC):
    xEffectiveSource = (xGrid - u * t) % L
    return phi0Func(xEffectiveSource, xStart=xStartIC, xEnd=xEndIC)

# --- 1. Simulation Parameters ---
lengthDomainUpwind = 1.0
numXPointsUpwind = 101
dxStepUpwind = lengthDomainUpwind / (numXPointsUpwind - 1)
xGridUpwind = np.linspace(0, lengthDomainUpwind, numXPointsUpwind)

advectionVelocityUpwind = 1.0 # Advection velocity (MUST BE > 0 for this FTBS)
courantNumberUpwind = 0.8     # Courant number (should be <= 1 for stability)
deltaTimeUpwind = courantNumberUpwind * dxStepUpwind / advectionVelocityUpwind # u is positive
timeFinalUpwind = 0.5
numTimeStepsUpwind = int(timeFinalUpwind / deltaTimeUpwind)

print(f"\nUpwind (FTBS) - Domain: L={lengthDomainUpwind}, Nx={numXPointsUpwind}, dx={dxStepUpwind:.4f}")
print(f"Upwind (FTBS) - Velocity: u={advectionVelocityUpwind}, CFL={courantNumberUpwind:.2f}, dt={deltaTimeUpwind:.4e}, Time={timeFinalUpwind:.3f}, Steps={numTimeStepsUpwind}")

# --- 2. Initial Condition ---
xInitialStartUpwind = 0.1 # Adjusted IC position
xInitialEndUpwind = 0.4
phiCurrentUpwind = initialConditionSquareWave(xGridUpwind, xStart=xInitialStartUpwind, xEnd=xInitialEndUpwind)
phiInitialUpwind = phiCurrentUpwind.copy()

# --- 3. Time-stepping Loop for Upwind (FTBS) ---
phiNewUpwind = np.zeros_like(phiCurrentUpwind) # Array for solution at n+1

# Define the number of unique intervals for modulo arithmetic
if numXPointsUpwind > 1:
    num_unique_intervals = numXPointsUpwind - 1
else:
    num_unique_intervals = 1 # Avoid division by zero for a single point grid

for nLoopIndex in range(numTimeStepsUpwind):
    phiOldStepUpwind = phiCurrentUpwind.copy() # Values from time level 'n'

    # Loop over the unique physical nodes 0 to Nx-2
    for iNode in range(numXPointsUpwind - 1):

        # Determine index for the upwind neighbor phi[i-1] with periodicity
        # Modulo is applied to the number of unique intervals
        iMinus1 = (iNode - 1) % num_unique_intervals

        phiNewUpwind[iNode] = phiOldStepUpwind[iNode] - \
                              courantNumberUpwind * (phiOldStepUpwind[iNode] - phiOldStepUpwind[iMinus1])

    # Synchronize the alias node at the end of the step
    if numXPointsUpwind > 1:
        phiNewUpwind[numXPointsUpwind - 1] = phiNewUpwind[0]

    phiCurrentUpwind = phiNewUpwind.copy() # Update solution for the next time step

# --- 4. Analytical Solution for Comparison ---
phiAnalyticalFinalUpwind = analyticalSolutionPeriodic(
    xGridUpwind, timeFinalUpwind, advectionVelocityUpwind,
    lengthDomainUpwind, initialConditionSquareWave,
    xStartIC=xInitialStartUpwind, xEndIC=xInitialEndUpwind)

# --- 5. Plotting Upwind (FTBS) Result ---
plt.figure(figsize=(10, 6))
plt.plot(xGridUpwind, phiInitialUpwind, 'k-', lw=1.5, label='Initial Condition (t=0)')
plt.plot(xGridUpwind, phiAnalyticalFinalUpwind, 'g--', lw=2, label='Analytical Solution')
plt.plot(xGridUpwind, phiCurrentUpwind, 'b.-', markersize=5,
         label=f'Upwind (FTBS, CFL={courantNumberUpwind:.2f})')
plt.xlabel('Position x')
plt.ylabel('$\phi$')
plt.title(f'Upwind (FTBS) Scheme for Advection (t={timeFinalUpwind:.2f})')
plt.legend()
plt.grid(True)
plt.ylim(-0.1, 1.1)
plt.show()

**Initial Observation:** The Upwind (FTBS) scheme successfully advects the profile without the catastrophic instabilities of FTCS (provided the CFL condition $C \le 1$ is met). However, a new numerical artifact is evident: **numerical diffusion**, which smears the sharp edges of the square wave.

## Chapter 7 Exercises

These exercises are designed to help you implement and test the finite difference schemes for the 1D linear advection equation, explore the CFL condition, and observe common numerical artifacts. Base your work on the Python scripts and concepts presented in this chapter. For all simulations, unless otherwise specified, use periodic boundary conditions.

### E7.1: FTCS Scheme - Detailed Instability Exploration

Refer to the Python script for the FTCS scheme (e.g., based on Listing 7.2). Use an initial square wave profile (e.g., $\phi=1$ for $0.1L \le x \le 0.4L$ and $\phi=0$ elsewhere on a domain $L=1.0$). Set advection velocity $u=1.0$.

1.  **Varying Courant Number ($C$):**
    Set $N_x=101$ (number of spatial points). Calculate $\Delta x$. Then, run simulations up to $t_{final}=0.05$ (a short time, as FTCS is unstable) for the following Courant numbers: $C = 0.1, 0.5, 1.0$.
    - For each $C$, calculate the required $\Delta t = C \Delta x / u$.
    - Plot the numerical solution at $t_{final}$ alongside the initial condition and the analytical solution.
    - Describe how the instability manifests for different $C$ values. Does it always look the same? Does it appear faster or slower?

2.  **Effect of Spatial Resolution ($\Delta x$):**
    Fix the Courant number at a value you found to be quickly unstable, e.g., $C=0.5$. Now, vary the spatial resolution:
    - Run with $N_x = 51$. Calculate the new $\Delta x$ and $\Delta t$ (to keep $C=0.5$). Simulate up to $t_{final}=0.05$.
    - Run with $N_x = 201$. Calculate the new $\Delta x$ and $\Delta t$. Simulate up to $t_{final}=0.05$.
    Plot the results. Does changing $\Delta x$ (while keeping $C$ constant) prevent the instability of the FTCS scheme for pure advection?

In [ ]:
# E7.1: FTCS Scheme Instability Exploration
import numpy as np
import matplotlib.pyplot as plt

# Re-using the functions from the chapter
def initialConditionSquareWave(x, xStart, xEnd):
    phi0 = np.zeros_like(x)
    phi0[(x >= xStart) & (x <= xEnd)] = 1.0
    return phi0

def analyticalSolutionPeriodic(xGrid, t, u, L, phi0Func, xStartIC, xEndIC):
    xEffectiveSource = (xGrid - u * t) % L
    return phi0Func(xEffectiveSource, xStart=xStartIC, xEnd=xEndIC)

def run_ftcs_simulation(numXPoints, courantNumber):
    lengthDomain = 1.0
    advectionVelocity = 1.0
    timeFinal = 0.05
    xStartIC = 0.1
    xEndIC = 0.4

    dxStep = lengthDomain / (numXPoints - 1)
    deltaTime = courantNumber * dxStep / advectionVelocity
    numTimeSteps = int(timeFinal / deltaTime)

    print(f"\nRunning FTCS with Nx={numXPoints}, C={courantNumber:.2f}")

    xGrid = np.linspace(0, lengthDomain, numXPoints)
    phiInitial = initialConditionSquareWave(xGrid, xStart=xStartIC, xEnd=xEndIC)
    phiAnalytic = analyticalSolutionPeriodic(xGrid, timeFinal, advectionVelocity, lengthDomain,
                                           initialConditionSquareWave, xStartIC, xEndIC)

    phiCurrent = phiInitial.copy()
    phiNew = np.zeros_like(phiCurrent)

    if numTimeSteps > 0:
        num_unique_intervals = numXPoints - 1
        for _ in range(numTimeSteps):
            phiOld = phiCurrent.copy()
            for i in range(numXPoints):
                i_minus_1 = (i - 1) % num_unique_intervals
                i_plus_1 = (i + 1) % num_unique_intervals
                phiNew[i] = phiOld[i] - (courantNumber / 2.0) * (phiOld[i_plus_1] - phiOld[i_minus_1])

            if numXPoints > 1: phiNew[-1] = phiNew[0]
            phiCurrent = phiNew.copy()

    return xGrid, phiInitial, phiCurrent, phiAnalytic

# 1. Varying Courant Number
courant_numbers = [0.1, 0.5, 1.0]
plt.figure(figsize=(12, 5))
plt.suptitle('E7.1, Task 1: Varying Courant Number (Nx=101)')
for i, C in enumerate(courant_numbers):
    ax = plt.subplot(1, len(courant_numbers), i + 1)
    x, T0, Tf, Ta = run_ftcs_simulation(101, C)
    ax.plot(x, T0, 'k:', label='Initial')
    ax.plot(x, Ta, 'r--', label='Analytical')
    ax.plot(x, Tf, 'b.-', markersize=4, label='Numerical')
    ax.set_title(f'C = {C:.1f}')
    ax.set_xlabel('x')
    ax.grid(True)
    ax.set_ylim(-1, 2)
    if i == 0: ax.set_ylabel('$\\phi$')
    if i == 2: ax.legend()
plt.tight_layout()
plt.show()

# 2. Effect of Spatial Resolution
nx_values = [51, 101, 201]
plt.figure(figsize=(12, 5))
plt.suptitle('E7.1, Task 2: Varying Spatial Resolution (C=0.5)')
for i, nx in enumerate(nx_values):
    ax = plt.subplot(1, len(nx_values), i + 1)
    x, T0, Tf, Ta = run_ftcs_simulation(nx, 0.5)
    ax.plot(x, T0, 'k:', label='Initial')
    ax.plot(x, Ta, 'r--', label='Analytical')
    ax.plot(x, Tf, 'b.-', markersize=4, label='Numerical')
    ax.set_title(f'Nx = {nx}')
    ax.set_xlabel('x')
    ax.grid(True)
    ax.set_ylim(-1, 2)
    if i == 0: ax.set_ylabel('$\\phi$')
    if i == 2: ax.legend()
plt.tight_layout()
plt.show()

### E7.2: Upwind Scheme - Impact of Courant Number and Numerical Diffusion

Refer to the Python script for the first-order Upwind (FTBS for $u>0$) scheme (e.g., based on Listing 7.3). Use an initial square wave profile (e.g., $\phi=1$ for $0.1L \le x \le 0.4L$, $\phi=0$ elsewhere, $L=1.0$) and $u=1.0$. Simulate up to $t_{final}=0.5$.

1.  **Varying Courant Number ($C$):**
    Set $N_x=101$. Run simulations for $C = 0.2, 0.5, 0.8, 1.0$.
    - Plot the numerical solution at $t_{final}$ for each $C$ value on the same graph, along with the initial condition and the analytical solution.
    - How does the amount of numerical diffusion (smearing of the square wave) change as $C$ varies within the stable range ($0 < C \le 1$)? Which value of $C$ gives the "sharpest" (least diffusive) result for the Upwind scheme?

2.  **Testing Stability Limit:**
    Now try $C = 1.01$ (or $C=1.1$). Plot the result. What happens when the CFL condition $C \le 1$ is violated for the Upwind scheme? Compare this to the FTCS instability.

3.  **Numerical Diffusion Coefficient (Conceptual):**
    The modified equation for the first-order upwind scheme includes a numerical diffusion term $D_{num} \frac{\partial^2 \phi}{\partial x^2}$ where $D_{num} = \frac{u \Delta x}{2}(1-C)$.
    - For a fixed $\Delta x$ and $u$, how does $D_{num}$ change as $C$ goes from near 0 to 1? Does this match your observations of smearing from part (1)?
    - What happens to $D_{num}$ if $C=1$? What does this imply for the accuracy of the Upwind scheme at $C=1$?

In [ ]:
# E7.2: Upwind Scheme Analysis
import numpy as np
import matplotlib.pyplot as plt

def run_upwind_simulation(courantNumber):
    lengthDomain = 1.0
    numXPoints = 101
    advectionVelocity = 1.0
    timeFinal = 0.5
    xStartIC = 0.1
    xEndIC = 0.4

    dxStep = lengthDomain / (numXPoints - 1)
    deltaTime = courantNumber * dxStep / advectionVelocity
    numTimeSteps = int(timeFinal / deltaTime)

    xGrid = np.linspace(0, lengthDomain, numXPoints)
    phiInitial = initialConditionSquareWave(xGrid, xStart=xStartIC, xEnd=xEndIC)
    phiAnalytic = analyticalSolutionPeriodic(xGrid, timeFinal, advectionVelocity, lengthDomain,
                                           initialConditionSquareWave, xStartIC, xEndIC)

    phiCurrent = phiInitial.copy()
    phiNew = np.zeros_like(phiCurrent)

    if numTimeSteps > 0:
        num_unique_intervals = numXPoints - 1
        for _ in range(numTimeSteps):
            phiOld = phiCurrent.copy()
            for i in range(numXPoints - 1):
                i_minus_1 = (i - 1) % num_unique_intervals
                phiNew[i] = phiOld[i] - courantNumber * (phiOld[i] - phiOld[i_minus_1])
            phiNew[-1] = phiNew[0]
            phiCurrent = phiNew.copy()

    return xGrid, phiInitial, phiCurrent, phiAnalytic

# 1. Varying Courant Number
courant_numbers = [0.2, 0.5, 0.8, 1.0]
plt.figure(figsize=(10, 6))
x, T0, _, Ta = run_upwind_simulation(courant_numbers[0]) # Run once to get initial/analytic
plt.plot(x, T0, 'k:', label='Initial Condition')
plt.plot(x, Ta, 'r--', label='Analytical Solution')
for C in courant_numbers:
    _, _, Tf, _ = run_upwind_simulation(C)
    plt.plot(x, Tf, '-', label=f'Upwind (C={C:.1f})')
plt.title('E7.2, Task 1: Upwind Scheme with Varying Courant Number')
plt.xlabel('x')
plt.ylabel('$\phi$')
plt.legend()
plt.grid(True)
plt.show()

# 2. Testing Stability Limit
C_unstable = 1.01
x, T0, Tf_unstable, Ta = run_upwind_simulation(C_unstable)
plt.figure(figsize=(10, 6))
plt.plot(x, T0, 'k:', label='Initial Condition')
plt.plot(x, Ta, 'r--', label='Analytical Solution')
plt.plot(x, Tf_unstable, 'b.-', markersize=4, label=f'Upwind (C={C_unstable})')
plt.title('E7.2, Task 2: Upwind Scheme Violating CFL Condition')
plt.xlabel('x')
plt.ylabel('$\\phi$')
plt.legend()
plt.grid(True)
plt.show()

### E7.3: Advecting Different Initial Profiles with Upwind Scheme

Using the Upwind (FTBS, $u>0$) scheme with $N_x=101$, $L=1.0$, $u=1.0$, and a stable Courant number (e.g., $C=0.8$), simulate the advection up to $t_{final}=0.5$ for the following initial conditions:
1. A narrow Gaussian pulse (e.g., use `initialConditionGaussian` from the notebook, perhaps with `x0=0.25`, `sigma=0.05`).
2. A sine wave, e.g., $\phi_0(x) = 0.5 + 0.5\sin(2\pi x/L)$.
3. A triangular pulse.

For each case, plot the initial condition, the analytical solution at $t_{final}$, and the numerical solution at $t_{final}$. Discuss how well the Upwind scheme preserves the shape of these different profiles. Does numerical diffusion affect smooth profiles differently from sharp ones?

In [ ]:
# E7.3: Advecting Different Initial Profiles
import numpy as np
import matplotlib.pyplot as plt

# --- Define Initial Conditions ---
def initialConditionGaussian(x, x0, sigma):
    return np.exp(-0.5 * ((x - x0) / sigma)**2)

def initialConditionSine(x, L):
    return 0.5 + 0.5 * np.sin(2 * np.pi * x / L)

def initialConditionTriangle(x, xStart, xPeak, xEnd):
    phi0 = np.zeros_like(x)
    # Rising part
    mask1 = (x >= xStart) & (x < xPeak)
    phi0[mask1] = (x[mask1] - xStart) / (xPeak - xStart)
    # Falling part
    mask2 = (x >= xPeak) & (x <= xEnd)
    phi0[mask2] = 1.0 - (x[mask2] - xPeak) / (xEnd - xPeak)
    return phi0

def run_upwind_for_ic(ic_func, L, u, t, Nx, C, **kwargs):
    dx = L / (Nx - 1)
    dt = C * dx / u
    Nt = int(t / dt)
    x = np.linspace(0, L, Nx)

    phi_initial = ic_func(x, **kwargs)
    phi_current = phi_initial.copy()
    phi_new = np.zeros_like(phi_current)

    num_unique_intervals = Nx - 1
    for _ in range(Nt):
        phi_old = phi_current.copy()
        for i in range(Nx - 1):
            i_minus_1 = (i - 1) % num_unique_intervals
            phi_new[i] = phi_old[i] - C * (phi_old[i] - phi_old[i_minus_1])
        phi_new[-1] = phi_new[0]
        phi_current = phi_new.copy()

    # Analytical
    x_shifted = (x - u * t) % L
    phi_analytic = ic_func(x_shifted, **kwargs)

    return x, phi_initial, phi_current, phi_analytic

# --- Simulation Parameters ---
L, u, t, Nx, C = 1.0, 1.0, 0.5, 101, 0.8

# --- Run and Plot for Each IC ---
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
plt.suptitle('E7.3: Advecting Different Initial Profiles with Upwind Scheme')

# 1. Gaussian Pulse
gauss_params = {'x0': 0.25, 'sigma': 0.05}
x_g, T0_g, Tf_g, Ta_g = run_upwind_for_ic(initialConditionGaussian, L, u, t, Nx, C, **gauss_params)
axs[0].plot(x_g, T0_g, 'k:', label='Initial')
axs[0].plot(x_g, Ta_g, 'r--', label='Analytical')
axs[0].plot(x_g, Tf_g, 'b-', label='Numerical')
axs[0].set_title('Gaussian Pulse')
axs[0].set_xlabel('x'); axs[0].set_ylabel('$\phi$'); axs[0].grid(True); axs[0].legend()

# 2. Sine Wave
sine_params = {'L': L}
x_s, T0_s, Tf_s, Ta_s = run_upwind_for_ic(initialConditionSine, L, u, t, Nx, C, **sine_params)
axs[1].plot(x_s, T0_s, 'k:', label='Initial')
axs[1].plot(x_s, Ta_s, 'r--', label='Analytical')
axs[1].plot(x_s, Tf_s, 'b-', label='Numerical')
axs[1].set_title('Sine Wave')
axs[1].set_xlabel('x'); axs[1].grid(True); axs[1].legend()

# 3. Triangular Pulse
tri_params = {'xStart': 0.1, 'xPeak': 0.25, 'xEnd': 0.4}
x_t, T0_t, Tf_t, Ta_t = run_upwind_for_ic(initialConditionTriangle, L, u, t, Nx, C, **tri_params)
axs[2].plot(x_t, T0_t, 'k:', label='Initial')
axs[2].plot(x_t, Ta_t, 'r--', label='Analytical')
axs[2].plot(x_t, Tf_t, 'b-', label='Numerical')
axs[2].set_title('Triangular Pulse')
axs[2].set_xlabel('x'); axs[2].grid(True); axs[2].legend()

plt.tight_layout()
plt.show()

### E7.4: Mass Conservation with Periodic Boundary Conditions

For the 1D linear advection equation $\frac{\partial \phi}{\partial t} + u \frac{\partial \phi}{\partial x} = 0$ with periodic boundary conditions, the total amount of $\phi$ in the domain, $M(t) = \int_0^L \phi(x,t) dx$, should be conserved over time if $u$ is constant. Numerically, we can approximate this integral as $M_n \approx \sum_{i=0}^{N_x-1} \phi_i^n \Delta x$.

**Tasks:**
1. Modify your Upwind (FTBS) script (using the square wave IC and $C=0.8$) to calculate and store this sum $M_n$ at each time step $n$.
2. At the end of the simulation, plot $M_n$ (or the percentage change from $M_0$) versus time.
3. Does the first-order Upwind scheme, as implemented, conserve the total mass $M_n$ exactly? If not, is the loss/gain significant?
4. **Conceptual:** Why might a scheme derived directly from the non-conservative form of the advection equation sometimes have issues with perfect discrete conservation, even with periodic BCs? (Hint: Think about how the sum $\sum (\phi_i^n - \phi_{i-1}^n)$ behaves over a periodic domain.)



In [ ]:
# E7.4: Mass Conservation
import numpy as np
import matplotlib.pyplot as plt

# --- Simulation Setup (similar to Upwind scheme) ---
L = 1.0
Nx = 101
u = 1.0
C = 0.8
t_final = 2.0 # Run for 2 full cycles

dx = L / (Nx - 1)
dt = C * dx / u
Nt = int(t_final / dt)

x = np.linspace(0, L, Nx)
phi_current = initialConditionSquareWave(x, xStart=0.1, xEnd=0.4)
phi_new = np.zeros_like(phi_current)

# --- Mass Calculation and Time-stepping ---
total_mass = []
time_steps = np.arange(0, Nt + 1) * dt

# Calculate initial mass
M0 = np.sum(phi_current[:-1]) * dx # Sum over unique intervals
total_mass.append(100.0) # Start at 100%

num_unique_intervals = Nx - 1
for n in range(Nt):
    phi_old = phi_current.copy()
    for i in range(Nx - 1):
        i_minus_1 = (i - 1) % num_unique_intervals
        phi_new[i] = phi_old[i] - C * (phi_old[i] - phi_old[i_minus_1])
    phi_new[-1] = phi_new[0]
    phi_current = phi_new.copy()

    # Calculate mass at current step and its percentage change from initial
    current_mass = np.sum(phi_current[:-1]) * dx
    percent_change = (current_mass / M0) * 100.0
    total_mass.append(percent_change)

# --- Plotting Mass Conservation ---
plt.figure(figsize=(10, 6))
plt.plot(time_steps, total_mass, 'b-')
plt.title('E7.4: Mass Conservation with FTBS Scheme')
plt.xlabel('Time')
plt.ylabel('Total Mass (% of Initial)')
plt.grid(True)
plt.ylim(99.9, 100.1) # Zoom in to see small changes
plt.show()

print(f"Final mass is {total_mass[-1]:.6f}% of the initial mass.")
# The first-order Upwind scheme for this form of the equation conserves mass very well,
# with errors only due to machine precision.